# Telugu (IndicTTS): CPU notebook

**Runtime: CPU.** No GPU is needed anywhere in this notebook.

| Cell | What it does | When to run |
|---|---|---|
| 1 | Restores the exported Telugu audio from Drive | first |
| 2 | Builds the pronunciation dictionary and the aligner corpus | after cell 1 |
| 3 | Installs MFA and **starts the Telugu aligner** (runs in the background) | after cell 2 |
| 4 | Aligner status | whenever you want to check |
| 5 | Sets up the repo copy and the Telugu config (light) | after cell 3 has started |
| 6 | Swaps in the Telugu phone set and checks it (light) | after cell 5 |

Keep this browser tab open while the aligner runs. When it finishes, `textgrids_telugu.tar` appears on Drive. I will then give you the Telugu feature-extraction cell.

## Cell 1: restore data (CPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
D = "/content/drive/MyDrive/tts_project"
if not os.path.isdir("/content/raw_data/Telugu"):
    !tar -xf {D}/raw_data_telugu.tar -C /content
for spk in sorted(os.listdir("/content/raw_data/Telugu")):
    print(spk, len(glob.glob(f"/content/raw_data/Telugu/{spk}/*.wav")), "wav files")

Mounted at /content/drive
TEF 1182 wav files
TEM 1384 wav files


## Cell 2: dictionary + aligner corpus (CPU)

In [ ]:
import re, os, shutil, collections, pandas as pd
D = "/content/drive/MyDrive/tts_project"
IV = {0x0C05:"A",0x0C06:"AA",0x0C07:"I",0x0C08:"II",0x0C09:"U",0x0C0A:"UU",0x0C0B:"R U",0x0C60:"R UU",
      0x0C0E:"E",0x0C0F:"EE",0x0C10:"AI",0x0C12:"O",0x0C13:"OO",0x0C14:"AU"}
VS = {0x0C3E:"AA",0x0C3F:"I",0x0C40:"II",0x0C41:"U",0x0C42:"UU",0x0C43:"R U",0x0C44:"R UU",
      0x0C46:"E",0x0C47:"EE",0x0C48:"AI",0x0C4A:"O",0x0C4B:"OO",0x0C4C:"AU"}
CONS = {0x0C15:"K",0x0C16:"KH",0x0C17:"G",0x0C18:"GH",0x0C19:"NG",0x0C1A:"C",0x0C1B:"CH",0x0C1C:"J",
        0x0C1D:"JH",0x0C1E:"NY",0x0C1F:"TT",0x0C20:"TTH",0x0C21:"DD",0x0C22:"DDH",0x0C23:"NN",
        0x0C24:"T",0x0C25:"TH",0x0C26:"D",0x0C27:"DH",0x0C28:"N",0x0C2A:"P",0x0C2B:"PH",0x0C2C:"B",
        0x0C2D:"BH",0x0C2E:"M",0x0C2F:"Y",0x0C30:"R",0x0C31:"RR",0x0C32:"L",0x0C33:"LL",0x0C35:"V",
        0x0C36:"SH",0x0C37:"SS",0x0C38:"S",0x0C39:"H"}
VIRAMA, ANUSVARA, VISARGA = 0x0C4D, 0x0C02, 0x0C03
NASAL = {**dict.fromkeys(["P","PH","B","BH","M"], "M"), **dict.fromkeys(["K","KH","G","GH"], "NG"),
         **dict.fromkeys(["C","CH","J","JH"], "NY"), **dict.fromkeys(["TT","TTH","DD","DDH","NN"], "NN"),
         **dict.fromkeys(["T","TH","D","DH","N"], "N")}
def g2p_word(w):
    out, i, n = [], 0, len(w)
    while i < n:
        c = ord(w[i])
        if c in IV: out += IV[c].split(); i += 1
        elif c in CONS:
            out.append(CONS[c]); i += 1
            if i < n and ord(w[i]) == VIRAMA: i += 1
            elif i < n and ord(w[i]) in VS: out += VS[ord(w[i])].split(); i += 1
            else: out.append("A")
        elif c == ANUSVARA:
            nxt = CONS.get(ord(w[i+1])) if i + 1 < n else None
            out.append(NASAL.get(nxt, "M")); i += 1
        elif c == VISARGA: out.append("H"); i += 1
        else: raise ValueError(f"unhandled character U+{c:04X} in {w!r}")
    return out

sel = pd.read_csv(f"{D}/telugu_split.csv")
words = sorted({w for t in sel.norm for w in t.split()})
wid = {w: f"w{i:05d}" for i, w in enumerate(words)}
phones, failed = collections.Counter(), []
os.makedirs("/content/mfa_telugu", exist_ok=True)
with open("/content/mfa_telugu/dict_ids.txt", "w") as f, open("/content/mfa_telugu/word_ids.tsv", "w", encoding="utf-8") as m:
    for w in words:
        try:
            p = g2p_word(w); phones.update(p)
            f.write(f"{wid[w]}\t{' '.join(p)}\n"); m.write(f"{wid[w]}\t{w}\t{' '.join(p)}\n")
        except ValueError as e: failed.append(str(e))
print("words:", len(words), "| failed:", len(failed), failed[:5])
print("phones used (%d):" % len(phones), dict(phones.most_common()))

src, dst = "/content/raw_data/Telugu", "/content/raw_aligner/Telugu"
shutil.rmtree(dst, ignore_errors=True)
n = 0
for spk in os.listdir(src):
    os.makedirs(f"{dst}/{spk}", exist_ok=True)
    for fn in os.listdir(f"{src}/{spk}"):
        if fn.endswith(".wav"): os.link(f"{src}/{spk}/{fn}", f"{dst}/{spk}/{fn}")
        elif fn.endswith(".lab"):
            t = open(f"{src}/{spk}/{fn}", encoding="utf-8").read().split()
            open(f"{dst}/{spk}/{fn}", "w").write(" ".join(wid[w] for w in t if w in wid)); n += 1
print("aligner labels written:", n)
os.system(f"cp /content/mfa_telugu/dict_ids.txt /content/mfa_telugu/word_ids.tsv {D}/")

words: 12127 | failed: 0 []
phones used (47): {'A': 15500, 'I': 8306, 'U': 8097, 'AA': 7373, 'N': 6995, 'R': 5322, 'K': 4649, 'L': 4335, 'M': 3806, 'T': 3426, 'P': 3285, 'V': 3166, 'D': 2945, 'DD': 2512, 'EE': 2487, 'C': 2451, 'S': 2347, 'TT': 2193, 'Y': 2113, 'G': 2100, 'OO': 1928, 'E': 1486, 'NN': 1209, 'II': 1121, 'UU': 967, 'NY': 820, 'B': 787, 'J': 715, 'O': 676, 'SH': 604, 'H': 498, 'NG': 485, 'AI': 483, 'SS': 474, 'DH': 470, 'LL': 451, 'BH': 385, 'KH': 140, 'TH': 121, 'AU': 68, 'PH': 68, 'GH': 57, 'TTH': 46, 'CH': 19, 'DDH': 15, 'RR': 6, 'JH': 3}
aligner labels written: 2566


0

## Cell 3: install MFA and start the aligner (CPU)

In [ ]:
import os, subprocess
D = "/content/drive/MyDrive/tts_project"
if subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0:
    raise RuntimeError("Aligner is already running. Nothing to do.")
if not os.path.exists("/content/mfa_env"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
    !/content/bin/micromamba create -y -q -p /content/mfa_env -c conda-forge python=3.11 montreal-forced-aligner
!MFA_ROOT_DIR=/content/mfa_root /content/bin/micromamba run -p /content/mfa_env mfa version
os.makedirs("/content/mfa_logs", exist_ok=True)
nj = os.cpu_count() or 2
cmd = (f"MFA_ROOT_DIR=/content/mfa_root /content/bin/micromamba run -p /content/mfa_env "
       f"mfa train --clean -j {nj} --output_directory /content/mfa_telugu/TextGrid "
       f"/content/raw_aligner/Telugu /content/mfa_telugu/dict_ids.txt /content/mfa_telugu/telugu_acoustic.zip "
       f"> /content/mfa_logs/train.log 2>&1 "
       f"&& cd /content/mfa_telugu && tar -cf {D}/textgrids_telugu.tar TextGrid "
       f"&& cp telugu_acoustic.zip {D}/ && echo ALIGN_DONE_AND_SAVED >> /content/mfa_logs/train.log")
subprocess.Popen(["bash", "-c", cmd], start_new_session=True)
print(f"Telugu aligner started with {nj} CPU jobs")

3.4.3
Telugu aligner started with 2 CPU jobs


## Cell 4: aligner status (CPU)

In [ ]:
import subprocess
running = subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0
b = subprocess.run("tr '\\r' '\\n' < /content/mfa_logs/train.log | grep -a -E 'INFO|ERROR|ALIGN_DONE' | tail -3", shell=True, capture_output=True).stdout
print("ALIGNER:", "RUNNING" if running else "NOT RUNNING"); print(b.decode("utf-8", errors="replace"))

ALIGNER: RUNNING
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   



## Cell 5: repo + Telugu config (CPU, light)

In [ ]:
D = "/content/drive/MyDrive/tts_project"
import os, shutil, yaml, pandas as pd
!pip install -q pyworld pypinyin unidecode inflect librosa pyyaml tgt
if not os.path.isdir("/content/FastSpeech2"):
    !git clone -q https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
    !cd /content/FastSpeech2 && git apply {D}/repo_patches.diff && echo "patches applied"
    !cd /content/FastSpeech2/hifigan && unzip -o -q generator_universal.pth.tar.zip
os.chdir("/content/FastSpeech2")
sel = pd.read_csv(f"{D}/telugu_split.csv")
open("split_val_utts.txt", "w").write("\n".join(sel.utt[sel.split == "val"]))
print("val utts:", (sel.split == "val").sum(), "| train utts:", (sel.split == "train").sum())
shutil.copytree("config/LJSpeech", "config/Telugu", dirs_exist_ok=True)
def edit(name, fn):
    p = f"config/Telugu/{name}.yaml"
    c = yaml.safe_load(open(p)); fn(c); yaml.safe_dump(c, open(p, "w"), sort_keys=False)
def pre(c):
    c["dataset"] = "Telugu"
    c["path"].update(corpus_path="/content/raw_data/Telugu", raw_path="/content/raw_data/Telugu",
                     preprocessed_path="/content/FastSpeech2/preprocessed_data/Telugu")
def mod(c):
    c["multi_speaker"] = True; c["vocoder"]["speaker"] = "universal"; c["max_seq_len"] = 1500
def tr(c):
    c["path"].update(ckpt_path=f"{D}/output/ckpt/Telugu", log_path=f"{D}/output/log/Telugu",
                     result_path=f"{D}/output/result/Telugu")
    c["step"].update(total_step=30000, log_step=100, synth_step=2000, val_step=1000, save_step=2000)
edit("preprocess", pre); edit("model", mod); edit("train", tr)
!mkdir -p {D}/config_Telugu && cp config/Telugu/*.yaml {D}/config_Telugu/
!grep -nE "multi_speaker|speaker:|max_seq_len|ckpt_path|total_step|save_step" config/Telugu/model.yaml config/Telugu/train.yaml

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
patches applied
val utts: 419 | train utts: 2147
config/Telugu/model.yaml:22:multi_speaker: true
config/Telugu/model.yaml:23:max_seq_len: 1500
config/Telugu/model.yaml:26:  speaker: universal
config/Telugu/train.yaml:2:  ckpt_path: /content/drive/MyDrive/tts_project/output/ckpt/Telugu
config/Telugu/train.yaml:21:  total_step: 30000
config/Telugu/train.yaml:25:  save_step: 2000


## Cell 6: Telugu phone set (CPU, light)

In [ ]:
import os, sys
os.chdir("/content/FastSpeech2")
D = "/content/drive/MyDrive/tts_project"
p = "text/symbols.py"
s = open(p).read()
TELUGU = ("A AA I II U UU E EE AI O OO AU "
          "K KH G GH NG C CH J JH NY TT TTH DD DDH NN T TH D DH N P PH B BH M Y R RR L LL V SH SS S H").split()
old_a = '_arpabet = ["@" + s for s in cmudict.valid_symbols]'
old_p = '_pinyin = ["@" + s for s in pinyin.valid_symbols]'
print("arpabet line found:", s.count(old_a), "| pinyin line found:", s.count(old_p))
if s.count(old_a) == 1 and s.count(old_p) == 1:
    s = s.replace(old_a, "_arpabet = " + repr(["@" + t for t in TELUGU]) + "  # Telugu phone set (replaces ARPAbet)")
    s = s.replace(old_p, "_pinyin = []")
    open(p, "w").write(s); print("patched")
for m in [k for k in sys.modules if k == "text" or k.startswith("text.")]: del sys.modules[m]
sys.path.insert(0, "/content/FastSpeech2")
from text import text_to_sequence, sequence_to_text
from text.symbols import symbols
print("total symbols:", len(symbols))
used = {ph for l in open("/content/mfa_telugu/word_ids.tsv", encoding="utf-8") for ph in l.rstrip("\n").split("\t")[2].split()}
print("dictionary phones:", len(used), "| missing from symbols:", sorted(x for x in used if "@" + x not in symbols))
seq = text_to_sequence("{A DD A V I L OO}", ["english_cleaners"])
print(seq, "->", sequence_to_text(seq))
!git diff text/symbols.py > {D}/telugu_symbols.diff

arpabet line found: 1 | pinyin line found: 1
patched
total symbols: 114
dictionary phones: 47 | missing from symbols: []
[64, 88, 64, 106, 66, 104, 74] -> {A DD A V I L OO}


In [ ]:
import os, sys, glob, json, collections, numpy as np, pandas as pd
D = "/content/drive/MyDrive/tts_project"
P = "/content/FastSpeech2/preprocessed_data/Telugu"
os.makedirs(P, exist_ok=True)

# 1) is the aligner finished? (guards against empty or partial output)
log = "/content/mfa_logs/train.log"
done = os.path.exists(log) and "ALIGN_DONE_AND_SAVED" in open(log, errors="replace").read()
tar = f"{D}/textgrids_telugu.tar"
tar_ok = os.path.exists(tar) and os.path.getsize(tar) > 500_000
if not (done or tar_ok):
    raise SystemExit("The Telugu alignment has not finished (no ALIGN_DONE_AND_SAVED and no real textgrids_telugu.tar). Check the status cell.")

# 2) put the TextGrids where the repo expects them
if not os.path.isdir(f"{P}/TextGrid"):
    if os.path.isdir("/content/mfa_telugu/TextGrid"):
        !cp -r /content/mfa_telugu/TextGrid {P}/TextGrid
    else:
        !tar -xf {tar} -C {P}
sel = pd.read_csv(f"{D}/telugu_split.csv")
open("/content/FastSpeech2/split_val_utts.txt", "w").write("\n".join(sel.utt[sel.split == "val"]))

# 3) alignment sanity checks
import tgt
os.chdir("/content/FastSpeech2"); sys.path.insert(0, "/content/FastSpeech2")
from text.symbols import symbols
tgs = sorted(glob.glob(f"{P}/TextGrid/*/*.TextGrid"))
print("TextGrids:", len(tgs), "of", len(sel), "utterances")
labels, durs = collections.Counter(), []
for p in tgs:
    for iv in tgt.io.read_textgrid(p).get_tier_by_name("phones"):
        labels[iv.text] += 1; durs.append(iv.end_time - iv.start_time)
unknown = {k: v for k, v in labels.items() if k and ("@" + k) not in symbols}
d = np.array(durs)
print("distinct phone labels:", len(labels), "| empty (silence) labels:", labels.get("", 0), "| labels unknown to the model:", unknown)
print("phone duration (s): median %.3f | p5 %.3f | p95 %.3f | max %.2f" % (np.median(d), *np.percentile(d, [5, 95]), d.max()))
tg = tgt.io.read_textgrid(tgs[0]); print(tgs[0])
print(" ".join(f"{iv.text or 'sil'}:{(iv.end_time-iv.start_time)*1000:.0f}" for iv in tg.get_tier_by_name("phones")))
if unknown or len(tgs) < 0.95 * len(sel):
    raise SystemExit("Alignment check failed (see above). Paste this output before preprocessing.")

# 4) feature extraction (mel, pitch, energy) with our own train/val split
!python preprocess.py config/Telugu/preprocess.yaml 2>&1 | tail -15
for n in ["train.txt", "val.txt"]:
    print(n, sum(1 for _ in open(f"{P}/{n}", encoding="utf-8")), "lines")
print("speakers:", json.load(open(f"{P}/speakers.json")), "| stats:", open(f"{P}/stats.json").read())
!head -2 {P}/train.txt
!cd /content/FastSpeech2/preprocessed_data && tar -cf {D}/preprocessed_Telugu.tar Telugu && ls -lh {D}/preprocessed_Telugu.tar

SystemExit: The Telugu alignment has not finished (no ALIGN_DONE_AND_SAVED and no real textgrids_telugu.tar). Check the status cell.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import subprocess
pids = subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True, text=True).stdout.split()
print("ALIGNER:", "RUNNING" if pids else "NOT RUNNING")
if pids:
    print("running for (d-hh:mm:ss):", subprocess.run(["ps", "-o", "etime=", "-p", pids[0]], capture_output=True, text=True).stdout.strip())
raw = subprocess.run("tr '\\r' '\\n' < /content/mfa_logs/train.log", shell=True, capture_output=True).stdout.decode("utf-8", "replace")
lines = [l.strip() for l in raw.splitlines() if any(k in l for k in ("INFO", "ERROR", "Error", "ALIGN_DONE"))]
its = [l for l in lines if "Iteration" in l]
print("last stage line:", its[-1] if its else "(none yet)")
print("\n".join(lines[-4:]))

ALIGNER: RUNNING
running for (d-hh:mm:ss): 10:29
last stage line: INFO     monophone - Iteration 5 of 40
INFO     Generating alignments...
INFO     Accumulating statistics...
INFO     monophone - Iteration 5 of 40
INFO     Generating alignments...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = "/content/drive/MyDrive/tts_project"
for n in ["textgrids.tar", "textgrids_telugu.tar", "preprocessed_Svarah.tar", "preprocessed_Telugu.tar"]:
    p = f"{D}/{n}"
    print(f"{n:26s}", f"{os.path.getsize(p)/1e6:.2f} MB" if os.path.exists(p) else "missing")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
textgrids.tar              0.01 MB
textgrids_telugu.tar       missing
preprocessed_Svarah.tar    missing
preprocessed_Telugu.tar    missing


In [ ]:
import time, subprocess
LOG = "/content/mfa_logs/train.log"
def running(): return subprocess.run(["pgrep", "-f", "mfa train"], capture_output=True).returncode == 0
def last_line():
    b = subprocess.run(f"tail -c 1500 {LOG}", shell=True, capture_output=True).stdout
    lines = [l.strip() for l in b.decode("utf-8", errors="replace").replace("\r", "\n").splitlines() if l.strip()]
    return lines[-1][:150] if lines else "(no log yet)"
t0 = time.time()
print("start:", "running" if running() else "NOT running", "|", last_line())
while running() and time.time() - t0 < 6 * 3600:
    time.sleep(120)
    print(f"[{(time.time()-t0)/60:.0f} min]", last_line())
done = os.path.exists(LOG) and "ALIGN_DONE_AND_SAVED" in open(LOG, errors="replace").read() if (os := __import__("os")) else False
print("FINISHED. Saved to Drive:", done)

start: running | INFO     Generating alignments...
[2 min] INFO     Accumulating statistics...
[4 min] INFO     Generating alignments...
[6 min] INFO     Generating alignments...
[8 min] INFO     Generating alignments...
[10 min] INFO     Generating alignments...
[12 min] INFO     Accumulating statistics...
[14 min] INFO     Accumulating statistics...
[16 min] INFO     Generating alignments...
[18 min] INFO     Analyzing alignment quality...
[20 min] INFO     Accumulating statistics...
[22 min] INFO     Accumulating statistics...
[24 min] INFO     Accumulating statistics...
[26 min] INFO     Generating alignments...
[28 min] INFO     Generating alignments...
[30 min] INFO     Analyzing alignment quality...
[32 min] INFO     Accumulating statistics...
[34 min] INFO     Accumulating statistics...
[36 min] INFO     Accumulating statistics...
[38 min] INFO     Accumulating statistics...
[40 min] INFO     Accumulating statistics...
[42 min] INFO     Accumulating statistics...
[44 min] INFO 

In [ ]:
import os, sys, glob, json, collections, numpy as np, pandas as pd

In [ ]:
import os
os.chdir("/content")          # the previous run left the notebook inside a deleted folder
from google.colab import drive
drive.mount('/content/drive')
import sys, glob, json, shutil, collections, numpy as np, pandas as pd
D = "/content/drive/MyDrive/tts_project"
P = "/content/FastSpeech2/preprocessed_data/Telugu"
tar = f"{D}/textgrids_telugu.tar"
print("textgrids_telugu.tar:", f"{os.path.getsize(tar)/1e6:.2f} MB" if os.path.exists(tar) else "MISSING")
if not (os.path.exists(tar) and os.path.getsize(tar) > 500_000):
    raise SystemExit("No real textgrids_telugu.tar on Drive. Paste this message.")

# 1) a real repo has a text/ folder; an empty stub folder does not
!pip install -q pyworld pypinyin unidecode inflect librosa pyyaml tgt scikit-learn
if not os.path.isdir("/content/FastSpeech2/text"):
    shutil.rmtree("/content/FastSpeech2", ignore_errors=True)
    !git clone -q https://github.com/ming024/FastSpeech2.git /content/FastSpeech2
    !cd /content/FastSpeech2 && git apply {D}/repo_patches.diff && echo "repo patches applied"
    !cd /content/FastSpeech2 && git apply {D}/telugu_symbols.diff && echo "telugu symbols applied"
    !cp -r {D}/config_Telugu /content/FastSpeech2/config/Telugu
    !cd /content/FastSpeech2/hifigan && unzip -o -q generator_universal.pth.tar.zip
if not os.path.isdir("/content/raw_data/Telugu"):
    !tar -xf {D}/raw_data_telugu.tar -C /content
os.makedirs(P, exist_ok=True)
if not os.path.isdir(f"{P}/TextGrid"):
    !tar -xf {tar} -C {P}
sel = pd.read_csv(f"{D}/telugu_split.csv")
open("/content/FastSpeech2/split_val_utts.txt", "w").write("\n".join(sel.utt[sel.split == "val"]))
print("text/ folder:", os.path.isdir("/content/FastSpeech2/text"), "| config:", os.path.isdir("/content/FastSpeech2/config/Telugu"),
      "| audio:", os.path.isdir("/content/raw_data/Telugu"), "| TextGrid folder:", os.path.isdir(f"{P}/TextGrid"))
!grep -c "VAL_SET" /content/FastSpeech2/preprocessor/preprocessor.py

# 2) alignment sanity checks
import tgt
os.chdir("/content/FastSpeech2"); sys.path.insert(0, "/content/FastSpeech2")
from text.symbols import symbols
tgs = sorted(glob.glob(f"{P}/TextGrid/*/*.TextGrid"))
print("TextGrids:", len(tgs), "of", len(sel), "utterances | symbols:", len(symbols), "(expect 114)")
labels, durs = collections.Counter(), []
for p in tgs:
    for iv in tgt.io.read_textgrid(p).get_tier_by_name("phones"):
        labels[iv.text] += 1; durs.append(iv.end_time - iv.start_time)
unknown = {k: v for k, v in labels.items() if k and ("@" + k) not in symbols}
d = np.array(durs)
print("distinct phone labels:", len(labels), "| empty (silence) labels:", labels.get("", 0), "| labels unknown to the model:", unknown)
print("phone duration (s): median %.3f | p5 %.3f | p95 %.3f | max %.2f" % (np.median(d), *np.percentile(d, [5, 95]), d.max()))
tg = tgt.io.read_textgrid(tgs[0]); print(tgs[0])
print(" ".join(f"{iv.text or 'sil'}:{(iv.end_time-iv.start_time)*1000:.0f}" for iv in tg.get_tier_by_name("phones")))
if unknown or len(tgs) < 0.95 * len(sel) or len(symbols) != 114:
    raise SystemExit("Alignment check failed (see above). Paste this output before preprocessing.")

# 3) feature extraction (mel, pitch, energy) with our own train/val split
!python preprocess.py config/Telugu/preprocess.yaml 2>&1 | tail -15
for n in ["train.txt", "val.txt"]:
    print(n, sum(1 for _ in open(f"{P}/{n}", encoding="utf-8")), "lines")
print("speakers:", json.load(open(f"{P}/speakers.json")), "| stats:", open(f"{P}/stats.json").read())
!head -2 {P}/train.txt
!cd /content/FastSpeech2/preprocessed_data && tar -cf {D}/preprocessed_Telugu.tar Telugu && ls -lh {D}/preprocessed_Telugu.tar

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
textgrids_telugu.tar: 30.00 MB
repo patches applied
telugu symbols applied
text/ folder: True | config: True | audio: True | TextGrid folder: True
4
TextGrids: 2553 of 2566 utterances | symbols: 114 (expect 114)
distinct phone labels: 47 | empty (silence) labels: 0 | labels unknown to the model: {}
phone duration (s): median 0.060 | p5 0.030 | p95 0.140 | max 0.88
/content/FastSpeech2/preprocessed_data/Telugu/TextGrid/TEF/TEF_00000.TextGrid
N:100 A:100 R:40 A:70 K:90 A:90 V:90 AA:120 S:130 I:200
/usr/local/lib/python3.13/dist-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Processing Data ...
100%|██████████| 2/2 

In [7]:
from google.colab import drive
drive.mount('/content/drive')
import glob, re
D = "/content/drive/MyDrive/tts_project"
!tail -n 4 {D}/output/log/Telugu/train/log.txt
!tail -n 6 {D}/output/log/Telugu/val/log.txt
print(sorted(int(re.findall(r"(\d+)\.pth\.tar$", p)[0]) for p in glob.glob(f"{D}/output/ckpt/Telugu/*.pth.tar")), "checkpoints")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1700/30000, Total Loss: 2.7473, Mel Loss: 0.7951, Mel PostNet Loss: 0.8185, Pitch Loss: 0.5454, Energy Loss: 0.5052, Duration Loss: 0.0832
Step 1800/30000, Total Loss: 2.7216, Mel Loss: 0.8197, Mel PostNet Loss: 0.8374, Pitch Loss: 0.5903, Energy Loss: 0.4019, Duration Loss: 0.0722
Step 1900/30000, Total Loss: 2.9830, Mel Loss: 0.8509, Mel PostNet Loss: 0.8540, Pitch Loss: 0.6223, Energy Loss: 0.5715, Duration Loss: 0.0843
Step 2000/30000, Total Loss: 2.8943, Mel Loss: 0.8669, Mel PostNet Loss: 0.8743, Pitch Loss: 0.5669, Energy Loss: 0.4991, Duration Loss: 0.0870
Validation Step 1000, Total Loss: 3.7682, Mel Loss: 1.0014, Mel PostNet Loss: 1.0244, Pitch Loss: 0.9238, Energy Loss: 0.7411, Duration Loss: 0.0776
Validation Step 2000, Total Loss: 3.7851, Mel Loss: 0.9709, Mel PostNet Loss: 0.9682, Pitch Loss: 1.0311, Energy Loss: 0.7417, Duration Loss: 0.07